# 🛠️ Tool 2 — NumPy Avancé : Matrices 2D, Agrégations Multi-Axes, Broadcasting et Aléatoire

Bienvenue dans le second module d'outils du cours **Data Science & Python** !

Dans **Tool 1**, nous avons exploré les arrays à une dimension : création, vectorisation, slicing, copies et masques booléens pour auditer un signal temporel de consommation.

Dans ce second volet, nous découvrons les dimensions supérieures et les fonctionnalités indispensables pour préparer les données au **Machine Learning** et aux réseaux de neurones :
- Comment manipuler des **tableaux 2D (matrices)** et réorganiser leur géométrie avec `.reshape()`.
- Comment maîtriser le paramètre **`axis`** pour calculer des moyennes ou des sommes par ligne ou par colonne sans se tromper.
- Comment exploiter le **Broadcasting** pour standardiser des variables en une seule ligne.
- Comment simuler des distributions statistiques réalistes avec le générateur moderne **`default_rng`**.
- Comment lisser un signal bruité avec la convolution vectorisée `np.convolve`.

---

> 💡 **Pédagogie interactive** : Ce notebook contient des **QCM interactifs** et des cellules de vérification automatique `check()` pour tester votre code au fil de l'eau.  
> 👉 `Shift + Entrée` pour exécuter chaque cellule.

In [ ]:
# Programme 1 — Initialisation & Outils interactifs (QCM et validation)
import numpy as np
import matplotlib.pyplot as plt
import random as _random_qcm
import ipywidgets as w
from IPython.display import display

# Relances humoristiques adaptées
_RELANCES_HUMOUR = [
    "Erreur sur l'axe : tu as confondu axis=0 et axis=1 !",
    "Erreur de dimension (BroadcastError) dans ton raisonnement !",
    "Attention à la forme : vérifie bien avec .shape !",
    "Pas d'inquiétude, même les créateurs de Scikit-Learn ont dû relire la doc de reshape un jour.",
    "Retente ta chance, le tableau 2D t'attend !"
]

def qcm(question, options, correct, explication=""):
    """
    Affiche un QCM interactif avec ipywidgets.
    correct : index (int) ou liste d'index de la/des bonne(s) réponse(s).
    """
    if isinstance(correct, int):
        correct = [correct]

    boutons = w.RadioButtons(options=options, layout=w.Layout(width='100%'))
    sortie  = w.Output()
    titre   = w.HTML(f"<b>❓ {question}</b>")

    def verifier(change):
        idx = options.index(change['new'])
        sortie.clear_output()
        with sortie:
            if idx in correct:
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#d4edda;color:#155724;border-left:5px solid #28a745'>"
                    f"✅ <b>Bravo, c'est exact !</b><br>{explication}</div>"
                ))
            else:
                humour = _random_qcm.choice(_RELANCES_HUMOUR)
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#f8d7da;color:#721c24;border-left:5px solid #dc3545'>"
                    f"❌ <b>Pas tout à fait...</b> {humour}<br><br>"
                    f"<b>Explication :</b> {explication}</div>"
                ))

    boutons.observe(verifier, names='value')
    display(w.VBox([titre, boutons, sortie]))

def check(nom, valeur, attendu, tol=1e-5):
    """
    Vérifie la valeur d'une variable ou d'un array calculé dans un exercice.
    Affiche un retour visuel vert ou rouge.
    """
    ok = False
    try:
        if isinstance(attendu, np.ndarray) and isinstance(valeur, np.ndarray):
            ok = valeur.shape == attendu.shape and np.allclose(valeur, attendu, atol=tol)
        elif isinstance(attendu, (int, float, np.integer, np.floating)) and isinstance(valeur, (int, float, np.integer, np.floating)):
            ok = abs(valeur - attendu) <= tol
        elif isinstance(attendu, (tuple, list)) and isinstance(valeur, (tuple, list)):
            ok = len(valeur) == len(attendu) and all(abs(v - a) <= tol for v, a in zip(valeur, attendu))
        else:
            ok = bool(valeur == attendu)
    except Exception:
        ok = False

    if ok:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#d4edda;color:#155724;border-left:4px solid #28a745'>"
            f"✅ <b>{nom}</b> est correct ! (Valeur : <code>{repr(valeur)}</code>)</div>"
        ))
    else:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#f8d7da;color:#721c24;border-left:4px solid #dc3545'>"
            f"❌ <b>{nom}</b> n'est pas encore la valeur attendue.<br>"
            f"Ta valeur : <code>{repr(valeur)}</code><br>"
            f"Valeur attendue : <code>{repr(attendu)}</code></div>"
        ))

print(f"✅ Environnement initialisé avec NumPy version {np.__version__}")
print("✅ Fonctions qcm() et check() prêtes à l'emploi.")

---
## 1 · Tableaux 2D (Matrices) et Réorganisation (`reshape`)

En Data Science, la majorité des jeux de données sont tabulaires :
- **Lignes** : les observations ou échantillons (instants de temps, clients, jours).
- **Colonnes** : les variables ou caractéristiques / *features* (tension, courant, température).

### 1.1 Indexation 2D : `[ligne, colonne]`
NumPy utilise une syntaxe élégante à virgule :
```python
mat[0, 2]     # Ligne 0, Colonne 2
mat[1, :]     # Toute la ligne 1
mat[:, 0]     # Toute la colonne 0
mat[0:2, 1:3] # Sous-matrice : lignes 0 à 1, colonnes 1 à 2
```

### 1.2 Réorganiser les dimensions avec `.reshape()`
La méthode `.reshape(n_lignes, n_colonnes)` permet de changer la forme d'un tableau sans modifier ses données ni allouer de nouvelle mémoire, à condition que le nombre total d'éléments soit conservé (`size = n_lignes * n_colonnes`).

💡 **L'astuce du `-1`** : vous pouvez indiquer `-1` pour une dimension, NumPy calculera automatiquement la taille correspondante !
```python
serie_24h = np.arange(24)              # 24 valeurs
matrice_4x6 = serie_24h.reshape(4, -1) # 4 lignes, NumPy déduit 6 colonnes
```

### 1.3 Ajouter un axe avec `np.newaxis` (ou `.reshape(-1, 1)`)
En Machine Learning (notamment avec **Scikit-Learn**), les modèles exigent toujours une matrice de features à 2 dimensions de forme `(n_échantillons, n_features)`.  
Pour transformer un vecteur 1D `(N,)` en colonne 2D `(N, 1)` :
```python
X = signal[:, np.newaxis]   # shape devient (N, 1)
```

In [2]:
# Programme 2 — Démonstration 2D et reshape
# Simulation de 12 heures de mesures de température
temp_1d = np.array([18.2, 18.5, 19.1, 20.4, 22.0, 23.5, 24.1, 23.8, 22.1, 20.5, 19.2, 18.6])

# Transformation en matrice de 3 jours x 4 créneaux horaires
mat_temp = temp_1d.reshape(3, 4)
print("Matrice 3x4 :\n", mat_temp)
print(f"Shape : {mat_temp.shape}")

# Extraction de la 1ère ligne (Jour 1 complet)
print("\nJour 1 :", mat_temp[0, :])

# Extraction de la 3ème colonne (Mesure de 14h pour tous les jours)
print("Créneau 3 pour tous les jours :", mat_temp[:, 2])

Matrice 3x4 :
 [[18.2 18.5 19.1 20.4]
 [22.  23.5 24.1 23.8]
 [22.1 20.5 19.2 18.6]]
Shape : (3, 4)

Jour 1 : [18.2 18.5 19.1 20.4]
Créneau 3 pour tous les jours : [19.1 24.1 19.2]


### ✏️ Exercice 1 — Remodeler un signal continu en matrice

On dispose de **24 mesures horaires** de température stockées dans `releve_24h`.
1. Remodelez ce vecteur en une matrice 2D `mat_jours` de **4 jours** et **6 tranches horaires** par jour (utilisez `.reshape(4, 6)`).
2. Extrayez dans la variable `midi_tous_les_jours` la 4ème tranche horaire (indice colonne `3`) pour l'ensemble des 4 jours.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
mat_jours = releve_24h.reshape(4, 6)
midi_tous_les_jours = mat_jours[:, 3]
```
</details>

In [ ]:
# Programme 3 — Exercice 1
releve_24h = np.array([
    15, 16, 19, 22, 20, 17,  # Jour 1
    14, 15, 18, 24, 21, 16,  # Jour 2
    16, 17, 20, 25, 22, 18,  # Jour 3
    15, 16, 19, 23, 19, 16   # Jour 4
])

# Remplacez les '...' par votre code :
mat_jours = ...
midi_tous_les_jours = ...

In [ ]:
# Programme 4 — Test Exercice 1
check("mat_jours shape", mat_jours.shape if hasattr(mat_jours, 'shape') else None, (4, 6))
check("midi_tous_les_jours", midi_tous_les_jours, np.array([22, 24, 25, 23]))

---
## 2 · Agrégations statistiques et le paramètre `axis`

NumPy fournit des méthodes d'agrégation ultra-rapides :
- `np.sum()`, `np.mean()`, `np.std()`, `np.var()`
- `np.min()`, `np.max()`, `np.median()`
- `np.argmin()`, `np.argmax()` : renvoie l'**indice** (la position) de la valeur minimale ou maximale !

### 🧭 Comprendre `axis=0` vs `axis=1` une bonne fois pour toutes

Sur une matrice 2D, appliquer `np.mean(mat)` calcule la moyenne globale de tous les nombres.  
Mais si on veut la moyenne par ligne ou par colonne, on utilise le paramètre `axis`.

> 💡 **La règle mnémotechnique infaillible de l'axe :**
> Pensez à `axis` comme l'axe qui est **écrasé (réduit)** lors du calcul !
>
> - `axis=0` : on suit la direction des lignes (verticale $\downarrow$).  
>   On écrase les lignes pour obtenir **un résultat par colonne**.  
>   *(Exemple : température moyenne à midi sur l'ensemble des jours).*  
>
> - `axis=1` : on suit la direction des colonnes (horizontale $\rightarrow$).  
>   On écrase les colonnes pour obtenir **un résultat par ligne**.  
>   *(Exemple : température moyenne de chaque jour).*  

```text
                  axis=1  ──►  (on parcourt chaque ligne, résultat par ligne)
                     Col 0   Col 1   Col 2
          ┌───────┬───────┬───────┐
  Ligne 0 │  10   │  20   │  30   │  ──► Moyenne Ligne 0 = 20
          ├───────┼───────┼───────┤
  Ligne 1 │  40   │  50   │  60   │  ──► Moyenne Ligne 1 = 50
          └───────┴───────┴───────┘
  axis=0     ▼       ▼       ▼
    │     Moy=25  Moy=35  Moy=45
    ▼
(on parcourt verticalement les lignes, résultat par colonne)
```

### 🎯 Trouver la position des extrêmes : `argmin()` et `argmax()`

Alors que `np.max(data)` renvoie la **valeur** maximale, `np.argmax(data)` renvoie l'**indice** (la position) de cette valeur :
- **Avec `axis=0`** : indice de la ligne maximale pour chaque colonne (résultat de taille `nb_colonnes`).
- **Avec `axis=1`** : indice de la colonne maximale pour chaque ligne (résultat de taille `nb_lignes`).
- **Sans `axis` (recherche globale)** : par défaut, NumPy aplatit virtuellement la matrice en 1D et renvoie un **indice à plat** (*flat index*).
  - Pour retrouver la valeur associée, on utilise **`.ravel()`**, qui déroule la matrice en un vecteur 1D sans recopie mémoire : `data.ravel()[idx_max]`.
  - Pour convertir cet indice à plat en coordonnées 2D `(ligne, colonne)`, on utilise `np.unravel_index(idx_max, data.shape)`.


In [5]:
# Programme 5 — Démonstration de calculs avec axis
data = np.array([
    [10, 20, 30],
    [40, 50, 60]
])

print("Moyenne globale :", data.mean())
print("Moyenne par colonne (axis=0) :", data.mean(axis=0), " -> shape :", data.mean(axis=0).shape)
print("Moyenne par ligne   (axis=1) :", data.mean(axis=1), " -> shape :", data.mean(axis=1).shape)

print("\n--- Recherche d'indices avec argmax ---")
# 1. Par axe (le plus utile sur une matrice)
print("Indice de la ligne max par colonne (axis=0) :", data.argmax(axis=0))
print("Indice de la col max par ligne   (axis=1) :", data.argmax(axis=1))

# 2. Globalement (sans axis) : renvoie l'indice à plat dans le tableau déroulé
idx_max_plat = np.argmax(data)  # Indice dans le tableau aplati [10, 20, 30, 40, 50, 60]
valeur_max = data.ravel()[idx_max_plat]  # .ravel() aplatit la matrice en vecteur 1D
coord_2d = np.unravel_index(idx_max_plat, data.shape)  # Conversion en (ligne, colonne)

print(f"Indice à plat du max : {idx_max_plat} -> valeur = {valeur_max}")
print(f"Coordonnées 2D du max (ligne, col) : {coord_2d}")


Moyenne globale : 35.0
Moyenne par colonne (axis=0) : [25. 35. 45.]  -> shape : (3,)
Moyenne par ligne   (axis=1) : [20. 50.]  -> shape : (2,)

--- Recherche d'indices avec argmax ---
Indice de la ligne max par colonne (axis=0) : [1 1 1]
Indice de la col max par ligne   (axis=1) : [2 2]
Indice à plat du max : 5 -> valeur = 60
Coordonnées 2D du max (ligne, col) : (1, 2)


In [6]:
# Programme 6 — QCM sur axis
qcm(
    "Dans une matrice X de taille (100, 5) représentant 100 maisons et 5 capteurs chacune, que retourne X.mean(axis=0) ?",
    [
        "Un vecteur de 100 valeurs (la moyenne des 5 capteurs pour chaque maison).",
        "Un vecteur de 5 valeurs (la moyenne de chaque capteur à travers les 100 maisons).",
        "Un scalaire unique (la moyenne générale de tout le quartier).",
        "Une matrice carrée (5, 5) de covariance."
    ],
    correct=1,
    explication="axis=0 réduit l'axe des 100 lignes. On moyenne verticalement chaque colonne, ce qui donne 5 moyennes (une par capteur)."
)

### ✏️ Exercice 2 — Statistiques par capteur

Considérons une matrice `releves` de 3 capteurs industriels (lignes) mesurés sur 5 cycles de production (colonnes).
1. Calculez la moyenne de chaque capteur sur ses 5 cycles dans `moyennes_capteurs` (réduction sur l'axe des colonnes, `axis=1`).
2. Calculez la consommation maximale observée sur chaque cycle dans `max_par_cycle` (réduction sur l'axe des capteurs, `axis=0`).

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
moyennes_capteurs = releves.mean(axis=1)
max_par_cycle = releves.max(axis=0)
```
</details>

In [ ]:
# Programme 7 — Exercice 2
releves = np.array([
    [120, 125, 122, 130, 128],   # Capteur A
    [ 80,  82,  85,  79,  81],   # Capteur B
    [210, 215, 205, 220, 218]    # Capteur C
])

# Remplacez les '...' par votre code :
moyennes_capteurs = ...
max_par_cycle = ...

In [ ]:
# Programme 8 — Test Exercice 2
check("moyennes_capteurs", moyennes_capteurs, releves.mean(axis=1))
check("max_par_cycle", max_par_cycle, releves.max(axis=0))

---
## 3 · Le Broadcasting : L'alignement automatique des formes

Que se passe-t-il si vous voulez soustraire la moyenne de chaque capteur à l'ensemble des mesures ?
La matrice `releves` a une forme `(3, 5)`, et le vecteur des moyennes a une forme `(3,)` ou `(3, 1)`.

Le **Broadcasting** est la capacité de NumPy à étendre virtuellement les petites dimensions sans dupliquer de données en mémoire pour rendre les formes compatibles.

### 📐 Les Règles du Broadcasting
NumPy compare les formes des deux tableaux dimension par dimension, **en partant de la droite** :
Deux dimensions sont compatibles si :
1. Elles sont **égales**, OU
2. L'une d'elles vaut **1** (cette dimension est alors étirée pour matcher l'autre).

### 🎯 Cas d'usage Data Science : Centrer / Normaliser des données (Z-Score)
Normaliser des données signifie soustraire la moyenne et diviser par l'écart-type :
$$Z = \frac{X - \mu}{\sigma}$$
Avec le broadcasting, cette opération complexe s'écrit en une seule ligne limpide !

In [9]:
# Programme 9 — Démonstration du Broadcasting & Standardisation
# Matrice de 4 échantillons (lignes) et 3 caractéristiques (colonnes)
X = np.array([
    [10.0, 100.0, 0.5],
    [12.0, 110.0, 0.6],
    [14.0, 105.0, 0.4],
    [11.0,  95.0, 0.7]
])

# Calcul de la moyenne et de l'écart-type par colonne (axis=0)
mu = X.mean(axis=0)    # shape : (3,)
sigma = X.std(axis=0)  # shape : (3,)

# Standardisation Z-Score grâce au broadcasting :
# X : (4, 3)
# mu :   (3,) -> broadcasté automatiquement sur les 4 lignes !
X_standardise = (X - mu) / sigma

print("Moyennes des colonnes :", np.round(mu, 2))
print("Écarts-types          :", np.round(sigma, 2))
print("\nDonnées standardisées (centrées en 0, écart-type=1) :\n", np.round(X_standardise, 2))
print("\nVérification des nouvelles moyennes :", np.round(X_standardise.mean(axis=0), 6))

Moyennes des colonnes : [ 11.75 102.5    0.55]
Écarts-types          : [1.48 5.59 0.11]

Données standardisées (centrées en 0, écart-type=1) :
 [[-1.18 -0.45 -0.45]
 [ 0.17  1.34  0.45]
 [ 1.52  0.45 -1.34]
 [-0.51 -1.34  1.34]]

Vérification des nouvelles moyennes : [ 0.  0. -0.]


---
## 4 · Nombres Aléatoires avec l'API Moderne (`default_rng`)

Générer des signaux synthétiques, ajouter du bruit réaliste ou initialiser les poids d'un réseau de neurones repose sur le hasard.

> 💡 **Bonne pratique moderne (NumPy $\ge$ 1.17) :**
> Évitez l'ancienne API `np.random.seed(42)` ou `np.random.rand()`.  
> Utilisez le générateur explicite **`np.random.default_rng(seed)`**.  
> Il est plus rapide, possède de meilleures propriétés cryptographiques/statistiques (générateur PCG64) et ne modifie pas l'état global du programme.

### Les distributions incontournables :
- `rng.uniform(low, high, size)` : tirage uniforme continu entre `low` et `high`.
- `rng.normal(loc, scale, size)` : distribution gaussienne (moyenne `loc`, écart-type `scale`). Idéal pour simuler le bruit de mesure.
- `rng.integers(low, high, size)` : entiers aléatoires entre `low` inclus et `high` exclu.
- `rng.choice(a, size=..., replace=...)` : tirage parmi une liste ou un tableau.

In [10]:
# Programme 10 — Démonstration de default_rng
# Création d'un générateur avec graine fixée pour reproductibilité
rng = np.random.default_rng(seed=42)

# 1. Bruit gaussien (moyenne 0, écart-type 2.5 W) sur 5 mesures
bruit = rng.normal(loc=0.0, scale=2.5, size=5)

# 2. Tirage d'entiers simulant des durées de cycle (entre 8 et 15 minutes)
durees = rng.integers(low=8, high=16, size=4)

print("Bruit gaussien (W) :", np.round(bruit, 2))
print("Durées de cycles   :", durees)

Bruit gaussien (W) : [ 0.76 -2.6   1.88  2.35 -4.88]
Durées de cycles   : [12 15 13 14]


---
## 5 · Mini-Projet : Synthèse & Analyse d'un Signal Énergétique avec Lissage

Mettons en pratique l'ensemble des concepts étudiés à travers une simulation concrète de traitement du signal !

### 🎯 Le Scénario
Nous voulons modéliser et analyser la puissance active $P(t)$ consommée par un appareil industriel sur **1000 secondes** (1 point par seconde) :
1. **Signal nominal** : une oscillation cyclique entre veille (~30 W) et pleine puissance (~150 W).
2. **Bruit physique** : un bruit gaussien de mesure $\pm 5\text{ W}$.
3. **Anomalies** : 3 surtensions parasites accidentelles ($> 200\text{ W}$).

Nous allons utiliser NumPy pour :
- Calculer le taux de marche (% du temps $> 60\text{ W}$).
- Détecter automatiquement les anomalies avec un masque booléen.
- Lisser le signal avec un filtre à moyenne glissante via la convolution vectorisée `np.convolve`.
- Tracer le résultat avec Matplotlib.

In [11]:
# Programme 11 — Simulation, traitement du signal et visualisation
# 1. Axe temporel (1000 secondes, 1 mesure/seconde)
t = np.linspace(0, 1000, 1000)

# 2. Signal carré périodique de base (période de 200 s : 80 s ON à 150 W, 120 s OFF à 30 W)
signal_base = np.where((t % 200) < 80, 150.0, 30.0)

# 3. Ajout de bruit gaussien (reproductible)
rng = np.random.default_rng(seed=123)
bruit = rng.normal(loc=0.0, scale=5.0, size=t.size)
signal_brut = signal_base + bruit

# 4. Injection de 3 anomalies ponctuelles à des instants précis
signal_brut[150] = 235.0
signal_brut[540] = 210.0
signal_brut[790] = 245.0

# 5. ANALYSE NUMPY :
# A) Détection des anomalies (> 180 W)
masque_anomalies = signal_brut > 180.0
indices_anomalies = np.where(masque_anomalies)[0]
nb_anomalies = indices_anomalies.size

# B) Taux de fonctionnement (% du temps où P > 60 W hors anomalies)
taux_marche_pct = (np.sum((signal_brut > 60.0) & ~masque_anomalies) / t.size) * 100

# C) Lissage par moyenne mobile sur 21 points avec np.convolve
fenetre = np.ones(21) / 21
signal_lisse = np.convolve(signal_brut, fenetre, mode='same')

print(f"=== Rapport d'Analyse NumPy ===")
print(f"Nombre de points analysés   : {signal_brut.size}")
print(f"Taux de fonctionnement ON   : {taux_marche_pct:.1f} %")
print(f"Anomalies détectées         : {nb_anomalies} pics aux indices {indices_anomalies.tolist()}")
print(f"Puissance max observée      : {signal_brut.max():.1f} W")

# 6. VISUALISATION MATPLOTLIB
plt.figure(figsize=(14, 5))
plt.plot(t, signal_brut, color='#6c757d', alpha=0.5, label='Signal brut bruité')
plt.plot(t, signal_lisse, color='#0d6efd', linewidth=2, label='Signal lissé (moyenne mobile 21 pts)')
plt.scatter(t[indices_anomalies], signal_brut[indices_anomalies], color='#dc3545', s=80, zorder=5, label='Anomalies détectées (>180 W)')

plt.axhline(60, color='#198754', linestyle='--', alpha=0.7, label='Seuil de marche (60 W)')
plt.title("⚡ Signal Électrique Industriel — Traitement et Détection NumPy", fontsize=14, fontweight='bold')
plt.xlabel("Temps (secondes)", fontsize=11)
plt.ylabel("Puissance (Watts)", fontsize=11)
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

=== Rapport d'Analyse NumPy ===
Nombre de points analysés   : 1000
Taux de fonctionnement ON   : 40.1 %
Anomalies détectées         : 3 pics aux indices [150, 540, 790]
Puissance max observée      : 245.0 W


/var/folders/kz/lyx_lm0532g9q8gdt228qdmw0000gn/T/ipykernel_38357/2364176393.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### ✏️ Défi final — Validation de l'analyse

À partir des résultats générés ci-dessus :
1. Affectez à la variable `nb_anom_trouvees` le nombre d'anomalies détectées.
2. Calculez la puissance médiane du signal brut dans `mediane_puissance` avec `np.median()`.
3. Validez vos réponses avec la cellule de test.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
nb_anom_trouvees = nb_anomalies
mediane_puissance = np.median(signal_brut)
```
</details>

In [ ]:
# Programme 12 — Défi final
# Remplacez les '...' par votre code :
nb_anom_trouvees = ...
mediane_puissance = ...

In [ ]:
# Programme 13 — Test Défi final
check("nb_anom_trouvees", nb_anom_trouvees, int(nb_anomalies))
check("mediane_puissance", mediane_puissance, float(np.median(signal_brut)), tol=0.1)

---
## 6 · Synthèse Générale & Aide-Mémoire (Cheat Sheet)

Félicitations ! Vous maîtrisez désormais l'ensemble des dimensions et des outils avancés de NumPy.

### 📋 Résumé des commandes incontournables

| Besoin | Fonction NumPy | Exemple |
|---|---|---|
| **Changer la forme** | `.reshape(lignes, cols)`, `np.newaxis` | `arr.reshape(-1, 1)` |
| **Inspecter la forme** | `.shape`, `.ndim`, `.size` | `mat.shape` → `(100, 5)` |
| **Statistiques par axe** | `axis=0` (vertical / par colonne), `axis=1` (horizontal / par ligne) | `mat.mean(axis=0)` |
| **Trouver l'indice min/max** | `np.argmin()`, `np.argmax()` | `arr.argmax()` |
| **Broadcasting** | Opérations entre formes compatibles | `(X - X.mean(axis=0)) / X.std(axis=0)` |
| **Hasard moderne** | `rng = np.random.default_rng(seed)` | `rng.normal(0, 1, 100)` |
| **Convolution / Moyenne mobile** | `np.convolve(signal, fenetre, mode='same')` | Filtre passe-bas / lissage |

---

### 🚀 Prochaine étape : Tool 3 — Pandas
NumPy est parfait pour manipuler des tableaux numériques bruts ultra-rapides.  
Mais dans la vraie vie, nos jeux de données comportent des **en-têtes de colonnes**, des **horodatages** (*timestamps*) et des types mixtes.  
C'est la mission de **Pandas** (`pd.DataFrame`), qui s'appuie directement sur NumPy tout en offrant un confort d'analyse tabulaire inégalé !